# Day 3 Statistical Analysis

Selected city: Stockholm, Sweden

This notebook is a readable companion to `src/run_day3_statistics.py`. The script is the reproducible source for the generated statistical report.

## Important Notes

- Calendar `price` and `adjusted_price` are 100% missing and are not used.
- Listing price tests use `listings_clean.price` and exclude missing prices.
- Statistical significance does not prove causation.
- Business significance is different from statistical significance.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from scipy import stats

project_root = Path.cwd()
db_path = project_root / 'data' / 'processed' / 'airbnb_stockholm.db'

with sqlite3.connect(db_path) as con:
    listings = pd.read_sql_query(
        '''
        SELECT room_type, price, host_is_superhost, review_scores_rating,
               number_of_reviews, neighbourhood_cleansed
        FROM listings_clean
        ''',
        con,
    )

listings.shape

## Example Test: Entire Homes vs Private Rooms

This test compares listing prices for two room types using the Mann-Whitney U test.

In [ ]:
priced = listings.dropna(subset=['price', 'room_type'])
entire = priced.loc[priced['room_type'] == 'Entire home/apt', 'price']
private = priced.loc[priced['room_type'] == 'Private room', 'price']

statistic, p_value = stats.mannwhitneyu(entire, private, alternative='two-sided')

pd.Series({
    'entire_home_count': entire.count(),
    'private_room_count': private.count(),
    'entire_home_median_price': entire.median(),
    'private_room_median_price': private.median(),
    'mann_whitney_u': statistic,
    'p_value': p_value,
})

Business interpretation: entire homes and private rooms are different products. A price difference supports segmenting pricing analysis by room type, but it does not prove room type alone causes the difference.

## Reproducible Command

Run from the project root:

```bash
python src/run_day3_statistics.py
```

Generated report: `reports/statistical_report.md`